#  Модуль 10. Feature Engineering — создание признаков

## Подробный конспект

### 10.1. Зачем создавать новые признаки

Модель — это только инструмент. Она не может «догадаться», что плотность населения важнее абсолютного населения, или что покупка в 3 часа ночи — подозрительна. **Она видит только то, что вы ей дадите.**

**Аналогия:** вы учите ребёнка отличать яблоки от груш. Если показываете только цвет, он запутается (бывают зелёные яблоки и груши). Но если добавите форму, запах, твёрдость — задача решается проще. Feature Engineering — это придумывание таких «правильных» признаков.

**Ключевой принцип:** качественные признаки + простая модель часто побеждают плохие признаки + сложную модель. На Kaggle и в индустрии именно FE отличает хорошие решения от средних.

### 10.2. Признаки из даты и времени

Дата в исходном виде (`2024-03-15 14:30:00`) — это строка или timestamp. Модель не понимает, что 15 марта — это весна, а 14:30 — середина дня. Нужно **распаковать** дату в компоненты.

#### Базовые компоненты

In [1]:
import pandas as pd

df = pd.DataFrame({
    'дата': pd.to_datetime(['2024-03-15 14:30:00', '2024-07-20 08:15:00', '2024-12-25 23:45:00'])
})

df['год'] = df['дата'].dt.year
df['месяц'] = df['дата'].dt.month
df['день'] = df['дата'].dt.day
df['день_недели'] = df['дата'].dt.dayofweek  # 0=Пн, 6=Вс
df['день_недели_имя'] = df['дата'].dt.day_name()
df['час'] = df['дата'].dt.hour
df['минута'] = df['дата'].dt.minute
df['квартал'] = df['дата'].dt.quarter
df['неделя_года'] = df['дата'].dt.isocalendar().week
df['сезон'] = df['месяц'].map({12: 'зима', 1: 'зима', 2: 'зима',
                               3: 'весна', 4: 'весна', 5: 'весна',
                               6: 'лето', 7: 'лето', 8: 'лето',
                               9: 'осень', 10: 'осень', 11: 'осень'})

df

,дата,год,месяц,день,день_недели,день_недели_имя,час,минута,квартал,неделя_года,сезон
0,2024-03-15 14:30:00,2024,3,15,4,Friday,14,30,1,11,весна
1,2024-07-20 08:15:00,2024,7,20,5,Saturday,8,15,3,29,лето
2,2024-12-25 23:45:00,2024,12,25,2,Wednesday,23,45,4,52,зима


#### Бинарные флаги (да/нет)

In [2]:
df['выходной'] = (df['дата'].dt.dayofweek >= 5).astype(int)
df['праздник'] = df['дата'].dt.date.isin([pd.Timestamp('2024-12-25').date(),
                                           pd.Timestamp('2024-01-01').date()]).astype(int)
df['ночь'] = ((df['час'] >= 23) | (df['час'] <= 5)).astype(int)
df['утро'] = ((df['час'] >= 6) & (df['час'] <= 11)).astype(int)
df['обед'] = ((df['час'] >= 12) & (df['час'] <= 14)).astype(int)

df

,дата,год,месяц,день,день_недели,день_недели_имя,час,минута,квартал,неделя_года,сезон,выходной,праздник,ночь,утро,обед
0,2024-03-15 14:30:00,2024,3,15,4,Friday,14,30,1,11,весна,0,0,0,0,1
1,2024-07-20 08:15:00,2024,7,20,5,Saturday,8,15,3,29,лето,1,0,0,1,0
2,2024-12-25 23:45:00,2024,12,25,2,Wednesday,23,45,4,52,зима,0,1,1,0,0


#### Циклические признаки (sin/cos)

Месяц, час, день недели — **циклические**. Декабрь (12) ближе к январю (1), чем к июню (6), но модель видит 12 > 6 > 1. Чтобы сохранить цикличность, используют синус и косинус.

In [3]:
import numpy as np

# Циклическое кодирование часа (24 часа)
df['час_sin'] = np.sin(2 * np.pi * df['час'] / 24)
df['час_cos'] = np.cos(2 * np.pi * df['час'] / 24)

# Циклическое кодирование дня недели (7 дней)
df['дн_нед_sin'] = np.sin(2 * np.pi * df['день_недели'] / 7)
df['дн_нед_cos'] = np.cos(2 * np.pi * df['день_недели'] / 7)

# Циклическое кодирование месяца (12 месяцев)
df['месяц_sin'] = np.sin(2 * np.pi * df['месяц'] / 12)
df['месяц_cos'] = np.cos(2 * np.pi * df['месяц'] / 12)

df

,дата,год,месяц,день,день_недели,день_недели_имя,час,минута,квартал,неделя_года,...,праздник,ночь,утро,обед,час_sin,час_cos,дн_нед_sin,дн_нед_cos,месяц_sin,месяц_cos
0,2024-03-15 14:30:00,2024,3,15,4,Friday,14,30,1,11,...,0,0,0,1,-0.500000,-0.866025,-0.433884,-0.900969,1.000000e+00,6.123234e-17
1,2024-07-20 08:15:00,2024,7,20,5,Saturday,8,15,3,29,...,0,0,1,0,0.866025,-0.500000,-0.974928,-0.222521,-5.000000e-01,-8.660254e-01
2,2024-12-25 23:45:00,2024,12,25,2,Wednesday,23,45,4,52,...,1,1,0,0,-0.258819,0.965926,0.974928,-0.222521,-2.449294e-16,1.000000e+00


**Почему это работает:** точки на окружности близки друг к другу, если углы близки. 23:00 и 01:00 дадут близкие пары (sin, cos), хотя числа 23 и 1 далеки.

#### Время с момента события (Time Since)

In [7]:
# Время с момента регистрации до сегодня
df['дата_регистрации'] = pd.to_datetime(['2020-01-15', '2023-06-10', '2024-01-01'])
сегодня = pd.Timestamp('2024-08-15')

df['дней_с_регистрации'] = (сегодня - df['дата_регистрации']).dt.days
df['месяцев_с_регистрации'] = (сегодня - df['дата_регистрации']).dt.days / 30.44
df

,дата,год,месяц,день,день_недели,день_недели_имя,час,минута,квартал,неделя_года,...,обед,час_sin,час_cos,дн_нед_sin,дн_нед_cos,месяц_sin,месяц_cos,дата_регистрации,дней_с_регистрации,месяцев_с_регистрации
0,2024-03-15 14:30:00,2024,3,15,4,Friday,14,30,1,11,...,1,-0.500000,-0.866025,-0.433884,-0.900969,1.000000e+00,6.123234e-17,2020-01-15,1674,54.993430
1,2024-07-20 08:15:00,2024,7,20,5,Saturday,8,15,3,29,...,0,0.866025,-0.500000,-0.974928,-0.222521,-5.000000e-01,-8.660254e-01,2023-06-10,432,14.191853
2,2024-12-25 23:45:00,2024,12,25,2,Wednesday,23,45,4,52,...,0,-0.258819,0.965926,0.974928,-0.222521,-2.449294e-16,1.000000e+00,2024-01-01,227,7.457293


#### Время до события (Time Until)

In [8]:
# Сколько дней до праздника
новый_год = pd.Timestamp('2025-01-01')
df['дней_до_НГ'] = (новый_год - df['дата']).dt.days
df

,дата,год,месяц,день,день_недели,день_недели_имя,час,минута,квартал,неделя_года,...,час_sin,час_cos,дн_нед_sin,дн_нед_cos,месяц_sin,месяц_cos,дата_регистрации,дней_с_регистрации,месяцев_с_регистрации,дней_до_НГ
0,2024-03-15 14:30:00,2024,3,15,4,Friday,14,30,1,11,...,-0.500000,-0.866025,-0.433884,-0.900969,1.000000e+00,6.123234e-17,2020-01-15,1674,54.993430,291
1,2024-07-20 08:15:00,2024,7,20,5,Saturday,8,15,3,29,...,0.866025,-0.500000,-0.974928,-0.222521,-5.000000e-01,-8.660254e-01,2023-06-10,432,14.191853,164
2,2024-12-25 23:45:00,2024,12,25,2,Wednesday,23,45,4,52,...,-0.258819,0.965926,0.974928,-0.222521,-2.449294e-16,1.000000e+00,2024-01-01,227,7.457293,6


### 10.3. Признаки из текста

Даже без сложного NLP можно извлечь полезную информацию.

#### Базовые мета-признаки

In [17]:
df = pd.DataFrame({
    'review': [
        'Great product, fast delivery!',
        'Did not like it',
        'Terrible, I do not recommend to anyone, complete nonsense and deception',
        None
    ]
})

# Заполним пропуски
df['review'] = df['review'].fillna('')

df['number_of_characters'] = df['review'].str.len()
df['number_of_words'] = df['review'].str.split().str.len()
df['number_of_sentences'] = df['review'].str.count(r'[.!?]+')
df['average_word_length'] = df['number_of_characters'] / (df['number_of_words'] + 0.001)

# Наличие ключевых слов
df['has_excellent'] = df['review'].str.contains('great|super|awesome|cool', case=False, regex=True).astype(int)
df['has_bad'] = df['review'].str.contains('terrible|bad|dislike|deception|nonsense', case=False, regex=True).astype(int)
df['has_exclamation'] = df['review'].str.contains('!').astype(int)
df['has_question'] = df['review'].str.contains(r'\?').astype(int)

df

,review,number_of_characters,number_of_words,number_of_sentences,average_word_length,has_excellent,has_bad,has_exclamation,has_question
0,"Great product, fast delivery!",29,4,1,7.248188,1,0,1,0
1,Did not like it,15,4,0,3.749063,0,0,0,0
2,"Terrible, I do not recommend to anyone, comple...",71,11,0,6.453959,0,1,0,0
3,,0,0,0,0.000000,0,0,0,0


#### TF-IDF и Bag of Words (краткое введение)

Если текст важен, можно превратить его в числовые признаки:

- **Bag of Words (CountVectorizer):** сколько раз каждое слово встретилось.
- **TF-IDF:** частота слова в документе, уменьшенная на то, как часто оно встречается во всех документах (редкие слова важнее).

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Для небольших задач — извлечь топ-N слов как признаки
vectorizer = TfidfVectorizer(max_features=50, stop_words='english')
text_features = vectorizer.fit_transform(df['review'])

# Превращаем в DataFrame
text_df = pd.DataFrame(
    text_features.toarray(),
    columns=[f'tfidf_{w}' for w in vectorizer.get_feature_names_out()]
)
text_df

,tfidf_complete,tfidf_deception,tfidf_delivery,tfidf_did,tfidf_fast,tfidf_great,tfidf_like,tfidf_nonsense,tfidf_product,tfidf_recommend,tfidf_terrible
0,0.000000,0.000000,0.5,0.000000,0.5,0.5,0.000000,0.000000,0.5,0.000000,0.000000
1,0.000000,0.000000,0.0,0.707107,0.0,0.0,0.707107,0.000000,0.0,0.000000,0.000000
2,0.447214,0.447214,0.0,0.000000,0.0,0.0,0.000000,0.447214,0.0,0.447214,0.447214
3,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000


> **Примечание:** полноценный NLP (BERT, Word2Vec) — это отдельная большая тема. Для табличного ML часто хватает TF-IDF или простых мета-признаков.

### 10.4. Математические комбинации признаков

Модель не умеет «догадываться», что доход на члена семьи важнее абсолютного дохода. Вы должны создать такой признак явно.

#### Арифметические комбинации

In [22]:
df = pd.DataFrame({
    'income': [50000, 80000, 120000],
    'expense': [40000, 30000, 90000],
    'area': [40, 65, 90],
    'weight': [70, 85, 60],
    'height': [175, 180, 165]
})

# Сумма и разность
df['savings'] = df['income'] - df['expense']

# Произведение и отношение
df['income_per_m2'] = df['income'] / df['area']
df['expense_share'] = df['expense'] / df['income']

# Индекс массы тела (BMI) — классический пример доменного знания
df['bmi'] = df['weight'] / (df['height'] / 100) ** 2

# Логарифм отношений
df['log_income_to_expense'] = np.log1p(df['income'] / (df['expense'] + 1))

df

,income,expense,area,weight,height,savings,income_per_m2,expense_share,bmi,log_income_to_expense
0,50000,40000,40,70,175,10000,1250.000000,0.800,22.857143,0.810916
1,80000,30000,65,85,180,50000,1230.769231,0.375,26.234568,1.299259
2,120000,90000,90,60,165,30000,1333.333333,0.750,22.038567,0.847292


#### Полиномиальные признаки

Если связь между признаком и таргетом нелинейная, можно добавить степени и произведения признаков.

In [31]:
from sklearn.preprocessing import PolynomialFeatures

X = df[['income', 'area']]

# Степени до 2: доход, площадь, доход ^ 2, доход * площадь, площадь ^ 2
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X)

feature_names = poly.get_feature_names_out(['income', 'area'])
poly_df = pd.DataFrame(X_poly, columns=feature_names)

poly_df

,income,area,income^2,income area,area^2
0,50000.0,40.0,2.500000e+09,2000000.0,1600.0
1,80000.0,65.0,6.400000e+09,5200000.0,4225.0
2,120000.0,90.0,1.440000e+10,10800000.0,8100.0


> **Осторожно:** полиномиальные признаки быстро разрастаются. Степень 2 для 10 признаков даст 65 новых столбцов.

### 10.5. Агрегационные признаки

Информация об одном объекте становится информативнее, если сравнить его с группой.

#### Группировка по категориям

In [34]:
df = pd.DataFrame({
    'city': ['Moscow', 'Moscow', 'SPb', 'SPb', 'Moscow'],
    'district': ['CAO', 'SAO', 'Center', 'Outskirts', 'CAO'],
    'price': [15_000_000, 8_000_000, 12_000_000, 6_000_000, 14_000_000],
    'area': [45, 35, 50, 30, 48]
})

# Средняя цена в городе (как «ожидаемая» цена для данного города)
average_by_city = df.groupby('city')['price'].transform('mean') # transform - сохраняем длину Series
df['average_price_city'] = average_by_city

# Отклонение от среднего по городу
df['deviation_from_city_avg'] = df['price'] - average_by_city

# Средняя цена за м² в городе
df['price_m2_by_city'] = df.groupby('city').apply(
    lambda x: x['price'] / x['area']
).reset_index(level=0, drop=True)

# Статистики по району внутри города
df['average_price_district'] = df.groupby(['city', 'district'])['price'].transform('mean')
df['object_count_in_district'] = df.groupby(['city', 'district'])['price'].transform('count')

df

,city,district,price,area,average_price_city,deviation_from_city_avg,price_m2_by_city,average_price_district,object_count_in_district
0,Moscow,CAO,15000000,45,1.233333e+07,2.666667e+06,333333.333333,14500000.0,2
1,Moscow,SAO,8000000,35,1.233333e+07,-4.333333e+06,228571.428571,8000000.0,1
2,SPb,Center,12000000,50,9.000000e+06,3.000000e+06,240000.000000,12000000.0,1
3,SPb,Outskirts,6000000,30,9.000000e+06,-3.000000e+06,200000.000000,6000000.0,1
4,Moscow,CAO,14000000,48,1.233333e+07,1.666667e+06,291666.666667,14500000.0,2


#### Оконные функции (Rolling / Expanding) для временных рядов

In [35]:
# Данные по дням
df_ts = pd.DataFrame({
    'date': pd.date_range('2024-01-01', periods=10, freq='D'),
    'sales': [100, 120, 90, 110, 130, 125, 140, 135, 150, 160]
})
df_ts = df_ts.set_index('date')

# Скользящее среднее (среднее за последние 3 дня)
df_ts['sales_rolling_3'] = df_ts['sales'].rolling(window=3).mean()

# Скользящее максимум/минимум/стандартное отклонение
df_ts['sales_rolling_max'] = df_ts['sales'].rolling(window=3).max()
df_ts['sales_rolling_std'] = df_ts['sales'].rolling(window=3).std()

# Нарастающий итог (expanding)
df_ts['sales_cumsum'] = df_ts['sales'].expanding().sum()
df_ts['sales_cummean'] = df_ts['sales'].expanding().mean()

# Сдвиг (lag features): продажи вчера, позавчера
df_ts['sales_lag_1'] = df_ts['sales'].shift(1)
df_ts['sales_lag_2'] = df_ts['sales'].shift(2)

df_ts

,sales,sales_rolling_3,sales_rolling_max,sales_rolling_std,sales_cumsum,sales_cummean,sales_lag_1,sales_lag_2
date,,,,,,,,
2024-01-01,100,NaN,NaN,NaN,100.0,100.000000,NaN,NaN
2024-01-02,120,NaN,NaN,NaN,220.0,110.000000,100.0,NaN
2024-01-03,90,103.333333,120.0,15.275252,310.0,103.333333,120.0,100.0
2024-01-04,110,106.666667,120.0,15.275252,420.0,105.000000,90.0,120.0
2024-01-05,130,110.000000,130.0,20.000000,550.0,110.000000,110.0,90.0
2024-01-06,125,121.666667,130.0,10.408330,675.0,112.500000,130.0,110.0
2024-01-07,140,131.666667,140.0,7.637626,815.0,116.428571,125.0,130.0
2024-01-08,135,133.333333,140.0,7.637626,950.0,118.750000,140.0,125.0
2024-01-09,150,141.666667,150.0,7.637626,1100.0,122.222222,135.0,140.0


**Важно:** при создании агрегационных признаков нельзя использовать **будущую информацию** (look-ahead bias). Для исторических данных — rolling, для прогноза — только прошлое.

### 10.6. Признаки на основе доменных знаний

Лучшие признаки рождаются из понимания предметной области.

#### Классические примеры

In [36]:
df = pd.DataFrame({
    'население_города': [12_000_000, 5_000_000, 1_000_000],
    'площадь_города_км2': [2500, 1400, 600],
    'вес_груза_кг': [500, 1200, 50],
    'объем_груза_м3': [2, 5, 0.5],
    'доход': [100000, 50000, 30000],
    'члены_семьи': [2, 4, 1]
})

# Плотность населения
df['плотность_населения'] = df['население_города'] / df['площадь_города_км2']

# Плотность груза (удельный вес)
df['удельный_вес_груза'] = df['вес_груза_кг'] / df['объем_груза_м3']

# Доход на члена семьи
df['доход_на_человека'] = df['доход'] / df['члены_семьи']

# Доход на душу населения (если бы это были регионы)
# df['ввп_на_душу'] = df['ввп'] / df['население']
df

,население_города,площадь_города_км2,вес_груза_кг,объем_груза_м3,доход,члены_семьи,плотность_населения,удельный_вес_груза,доход_на_человека
0,12000000,2500,500,2.0,100000,2,4800.000000,250.0,50000.0
1,5000000,1400,1200,5.0,50000,4,3571.428571,240.0,12500.0
2,1000000,600,50,0.5,30000,1,1666.666667,100.0,30000.0


**Как придумывать:**
- Спросите эксперта: «Как вы оцениваете этот объект?»
- Ищите отношения: «цена за единицу», «доля от целого», «интенсивность»
- Думайте в терминах физики, экономики, биологии предметной области.

### 10.7. Feature Interactions (Взаимодействия признаков)

Иногда один признак «работает» только в сочетании с другим. Например, высокий доход + ночные транзакции = подозрительно. По отдельности ни один из них не даёт такого сигнала.

In [37]:
df = pd.DataFrame({
    'доход': [50000, 200000, 50000, 200000],
    'ночная_транзакция': [0, 0, 1, 1],
    'таргет': [0, 0, 0, 1]  # мошенничество только при высоком доходе И ночи
})

# Явное взаимодействие
df['доход_x_ночь'] = df['доход'] * df['ночная_транзакция']

# Или категориальное взаимодействие
df['категория_дохода'] = pd.cut(df['доход'], bins=[0, 100000, 500000], labels=['низкий', 'высокий'])
df['комбо'] = df['категория_дохода'].astype(str) + '_' + df['ночная_транзакция'].astype(str)
# Получится: 'низкий_0', 'высокий_0', 'низкий_1', 'высокий_1'
df

,доход,ночная_транзакция,таргет,доход_x_ночь,категория_дохода,комбо
0,50000,0,0,0,низкий,низкий_0
1,200000,0,0,0,высокий,высокий_0
2,50000,1,0,50000,низкий,низкий_1
3,200000,1,1,200000,высокий,высокий_1


> **Совет:** не создавайте все возможные комбинации — это взрыв размерности. Создавайте те, о которых есть гипотеза.

### 10.8. Признаки-индикаторы (флаги)

Бинарные признаки, которые сигнализируют о наличии какого-то свойства. Часто они информативнее сырых чисел.

In [ ]:
df = pd.DataFrame({
    'возраст': [25, 45, 67, 12],
    'доход': [50000, 0, 120000, 30000],
    'дней_с_последней_покупки': [5, 180, 30, 400],
    'отзыв': ['', 'Хорошо', 'Ужас', 'Норм']
})

# Возрастные группы
df['пенсионер'] = (df['возраст'] >= 65).astype(int)
df['ребенок'] = (df['возраст'] <= 14).astype(int)
df['молодой_взрослый'] = ((df['возраст'] >= 18) & (df['возраст'] <= 30)).astype(int)

# Финансовые флаги
df['нулевой_доход'] = (df['доход'] == 0).astype(int)
df['высокий_доход'] = (df['доход'] > 100000).astype(int)

# Активность
df['давно_не_покупал'] = (df['дней_с_последней_покупки'] > 90).astype(int)
df['новый_клиент'] = (df['дней_с_последней_покупки'].isna()).astype(int)

# Текстовые флаги
df['отзыв_пустой'] = (df['отзыв'] == '').astype(int)
df['отзыв_негативный'] = df['отзыв'].str.contains('ужас|плохо|отврат', case=False, na=False).astype(int)

print(df[['пенсионер', 'нулевой_доход', 'давно_не_покупал', 'отзыв_негативный']])

#### Индикатор пропуска (повторение из Модуля 5, но в контексте FE)

In [ ]:
# Был ли пропуск в исходных данных (до заполнения)
df['доход_был_пропущен'] = df['доход'].isnull().astype(int)

### 10.9. Практический пример: полный цикл Feature Engineering

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

# Исходные «сырые» данные
df = pd.DataFrame({
    'user_id': range(1000),
    'дата_регистрации': pd.date_range('2022-01-01', periods=1000, freq='D'),
    'город': np.random.choice(['Москва', 'СПб', 'Казань', 'Новосибирск'], 1000),
    'возраст': np.random.randint(18, 70, 1000),
    'доход': np.random.lognormal(11, 0.5, 1000),
    'члены_семьи': np.random.randint(1, 6, 1000),
    'последняя_покупка': pd.date_range('2023-01-01', periods=1000, freq='H'),
    'трат_за_месяц': np.random.lognormal(9, 0.8, 1000)
})

# === 1. ВРЕМЕННЫЕ ПРИЗНАКИ ===
today = pd.Timestamp('2024-08-15')

df['дней_с_регистрации'] = (today - df['дата_регистрации']).dt.days
df['месяц_регистрации'] = df['дата_регистрации'].dt.month
df['квартал_регистрации'] = df['дата_регистрации'].dt.quarter
df['регистрация_лето'] = df['дата_регистрации'].dt.month.isin([6,7,8]).astype(int)

df['дней_с_покупки'] = (today - df['последняя_покупка']).dt.days
df['давно_не_покупал'] = (df['дней_с_покупки'] > 60).astype(int)

# === 2. ДОМЕННЫЕ ПРИЗНАКИ ===
df['доход_на_человека'] = df['доход'] / df['члены_семьи']
df['доля_трат'] = df['трат_за_месяц'] / (df['доход'] + 1)  # +1 чтобы избежать деления на 0

# === 3. ВОЗРАСТНЫЕ ФЛАГИ ===
df['молодой'] = (df['возраст'] <= 30).astype(int)
df['пенсионер'] = (df['возраст'] >= 60).astype(int)
df['семейный'] = (df['члены_семьи'] > 2).astype(int)

# === 4. АГРЕГАЦИИ ПО ГОРОДУ ===
city_stats = df.groupby('город').agg({
    'доход': 'median',
    'трат_за_месяц': 'mean',
    'возраст': 'mean'
}).rename(columns={
    'доход': 'медианный_доход_города',
    'трат_за_месяц': 'средние_траты_города',
    'возраст': 'средний_возраст_города'
})

df = df.merge(city_stats, left_on='город', right_index=True, how='left')

# Отклонение от города
df['доход_относительно_города'] = df['доход'] - df['медианный_доход_города']

# === 5. ВЗАИМОДЕЙСТВИЯ ===
df['доход_x_возраст'] = df['доход'] * df['возраст']
df['возраст_x_семья'] = df['возраст'] * df['члены_семьи']

# === 6. КАТЕГОРИАЛЬНЫЕ КОМБИНАЦИИ ===
df['город_возрастная_группа'] = df['город'] + '_' + pd.cut(
    df['возраст'], 
    bins=[0, 30, 45, 60, 100], 
    labels=['молодые', 'средние', 'зрелые', 'пожилые']
).astype(str)

print("Итоговые признаки:", df.shape[1])
print(df[['доход_на_человека', 'доля_трат', 'медианный_доход_города', 
          'доход_относительно_города', 'давно_не_покупал']].head())

### 10.10. Чек-лист для самопроверки

Перед переходом к Модулю 11 убедитесь, что вы:

- [ ] Понимаете, почему качественные признаки важнее сложности модели
- [ ] Умеете извлекать компоненты даты (год, месяц, день, час, день недели)
- [ ] Знаете, зачем создавать флаги выходных, праздников, времени суток
- [ ] Можете создать циклические признаки через sin/cos для часа и месяца
- [ ] Умеете считать «время с момента события» и «время до события»
- [ ] Можете извлечь базовые признаки из текста (длина, количество слов, ключевые слова)
- [ ] Знаете, что такое TF-IDF и когда он применяется
- [ ] Умеете создавать арифметические комбинации (сумма, разность, произведение, отношение)
- [ ] Можете создать полиномиальные признаки через `PolynomialFeatures`
- [ ] Умеете строить агрегационные признаки по группам (`.groupby().transform()`)
- [ ] Знаете, что такое rolling и expanding признаки для временных рядов
- [ ] Можете привести пример доменного признака (BMI, плотность, доход на человека)
- [ ] Понимаете идею feature interactions и можете создать комбинированный признак
- [ ] Умеете создавать бинарные индикаторы (флаги) по порогам

> **Переход к Модулю 11:** Теперь, когда вы умеете создавать признаки, пора научиться справляться с несбалансированными данными — ситуацией, когда один класс сильно преобладает над другим. Мы изучим методы сэмплирования, взвешивание и специальные метрики.